# Trabalho 2 — Validação de Dados, Testes Estatísticos e Fairness

Neste trabalho aplicamos uma suíte de validação de dados (baseada na Aula 3) e testes estatísticos e de fairness (baseados na Aula 4) sobre o dataset **Adult Census Income**.

- **Atributo sensível:** `sex` (Male / Female)
- **Target:** `income` (>50K / <=50K)

In [ ]:
import numpy as np
import pandas as pd
import pytest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Carregamento dos dados

In [ ]:
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df_raw = pd.read_csv(url, names=COLUMNS, sep=r",\s*", engine="python", na_values="?")
print(f"Formato original: {df_raw.shape}")

## 2. Funções de validação de dados (Aula 3)

In [ ]:
def validate_schema(df, expected_columns):
    missing = set(expected_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Colunas ausentes: {missing}")
    for col, expected_type in expected_columns.items():
        actual_type = str(df[col].dtype)
        if expected_type in ("object", "string", "str") and actual_type in ("object", "string", "str"):
            continue
        if actual_type != expected_type:
            raise ValueError(f"Coluna {col} com tipo {actual_type}, esperado {expected_type}")
    return True

def check_no_missing_values(df, columns):
    for col in columns:
        if df[col].isna().any():
            raise ValueError(f"Valores nulos na coluna {col}")
    return True

def check_range(df, column, lo, hi):
    if not df[column].dropna().between(lo, hi).all():
        raise ValueError(f"Coluna {column} com valores fora da faixa [{lo}, {hi}]")
    return True

## 3. Funções de estatística e fairness (Aula 4)

In [ ]:
def accuracy(y_true, y_pred):
    return (np.array(y_true) == np.array(y_pred)).mean()

def bootstrap_ci(y_true, y_pred, metric_fn=accuracy, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    n = len(y_true)
    metrics = [metric_fn(y_true[idx], y_pred[idx]) for idx in (rng.integers(0, n, n) for _ in range(n_boot))]
    return np.percentile(metrics, (1 - ci) / 2 * 100), np.percentile(metrics, (1 + ci) / 2 * 100)

def compare_models(y_true, pred_a, pred_b, metric_fn=accuracy, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true, pred_a, pred_b = np.array(y_true), np.array(pred_a), np.array(pred_b)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs.append(metric_fn(y_true[idx], pred_a[idx]) - metric_fn(y_true[idx], pred_b[idx]))
    return np.percentile(diffs, (1 - ci) / 2 * 100), np.percentile(diffs, (1 + ci) / 2 * 100)

def positive_rate(y_pred, group):
    y_p, grp = np.array(y_pred), np.array(group)
    return {g: float(y_p[grp == g].mean()) for g in sorted(set(grp))}

def check_demographic_parity(y_pred, group, max_gap=0.10):
    rates = positive_rate(y_pred, group)
    gap = max(rates.values()) - min(rates.values())
    if gap > max_gap:
        raise ValueError(f"Gap de paridade demográfica elevado: {gap:.4f} > {max_gap}")
    return True

## 4. Treinamento e avaliação dos modelos

In [ ]:
df_clean = df_raw.dropna().drop_duplicates().copy()
df_clean["target"] = (df_clean["income"] == ">50K").astype(int)

features_num = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
features_cat = ["workclass", "marital_status", "occupation", "relationship", "race", "sex"]

X_encoded = pd.get_dummies(df_clean[features_num + features_cat], drop_first=True)
y = df_clean["target"].values
sex = df_clean["sex"].values

X_tr, X_te, y_tr, y_te, sex_tr, sex_te = train_test_split(X_encoded.values, y, sex, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Modelo A (completo)
clf_a = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_s, y_tr)
pred_a = clf_a.predict(X_te_s)

# Modelo B (apenas numérico)
X_tr_num = X_tr[:, :len(features_num)]
X_te_num = X_te[:, :len(features_num)]
scaler_b = StandardScaler()
clf_b = LogisticRegression(max_iter=1000, random_state=42).fit(scaler_b.fit_transform(X_tr_num), y_tr)
pred_b = clf_b.predict(scaler_b.transform(X_te_num))

print(f"Acurácia Modelo A: {accuracy(y_te, pred_a):.4f}")
print(f"Acurácia Modelo B: {accuracy(y_te, pred_b):.4f}")

## 5. Análise dos resultados

### Validação dos Dados
- O dataset inicial contava com 32.561 linhas. Identificamos 2.399 linhas com dados ausentes (`?`) e 24 duplicatas.
- As faixas de valores (`age` entre 17 e 90, `hours_per_week` entre 1 e 99) atenderam às restrições.

### Teste Estatístico (Bootstrap & Comparação)
- O intervalo de confiança de 95% para a acurácia do Modelo A foi de **[84.0%, 85.5%]**.
- Na comparação entre o Modelo A e o Modelo B, o IC 95% da diferença de acurácia foi de **[5.2%, 7.1%]**. Como o intervalo não cruza o zero, a melhoria do modelo completo é estatisticamente significante.

### Fairness (Gênero)
- A taxa de predição positiva (`income > 50K`) foi de **26.1% para homens** e **8.1% para mulheres** (gap de 18.0%).
- O *Disparate Impact Ratio* é de **0.31**, bem abaixo do limiar de 0.80 da regra dos 80%.
- Conclusão: O modelo herda o viés histórico da base e precisaria de pós-processamento ou debiasing para mitigação antes de implantação.